In [1]:
# Install pyspark directly (no manual download)
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("KafkaStreaming") \
    .config("spark.jars.packages",
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true") \
    .getOrCreate()

In [3]:
from google.colab import drive
drive.mount('/content/drive')

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, expr

spark = SparkSession.builder.appName("EcommerceProject").getOrCreate()

# Load dataset
file_path = "/content/drive/MyDrive/Big Data Project/electronics_product.csv"

df = spark.read.csv(file_path, header=True, inferSchema=True)

df.show(5)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
+---+--------------------+-------------------+---------------+--------------------+--------------------+-------+-------------+--------------+------------+
|_c0|                name|      main_category|   sub_category|               image|                link|ratings|no_of_ratings|discount_price|actual_price|
+---+--------------------+-------------------+---------------+--------------------+--------------------+-------+-------------+--------------+------------+
|  0|Redmi 10 Power (P...|tv, audio & cameras|All Electronics|https://m.media-a...|https://www.amazo...|    4.0|          965|       ₹10,999|     ₹18,999|
|  1|OnePlus Nord CE 2...|tv, audio & cameras|All Electronics|https://m.media-a...|https://www.amazo...|    4.3|      113,956|       ₹18,999|     ₹19,999|
|  2|OnePlus Bullets Z...|tv, audio & cameras|All Electronics|https://m.media-a...|https://www.a

In [4]:
df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- image: string (nullable = true)
 |-- link: string (nullable = true)
 |-- ratings: string (nullable = true)
 |-- no_of_ratings: string (nullable = true)
 |-- discount_price: string (nullable = true)
 |-- actual_price: string (nullable = true)



In [5]:
df = df.select(
    col("name").alias("Name"),
    col("main_category").alias("Category"),
    col("sub_category").alias("Sub_Category"),
    col("ratings").alias("Rating"),
    col("no_of_ratings").alias("Rating_Count"),
    col("discount_price").alias("Price")
)

In [6]:
#Cleaning the Data

In [7]:
df = df.withColumn(
    "price_num",
    expr("try_cast(regexp_replace(Price, '[^0-9.]', '') as double)")
)

In [8]:
df = df.withColumn(
    "rating_num",
    expr("try_cast(regexp_replace(Rating, '[^0-9.]', '') as double)")
)

In [9]:
df = df.withColumn(
    "rating_count_num",
    expr("try_cast(regexp_replace(Rating_Count, '[^0-9]', '') as int)")
)

In [10]:
df = df.dropna(subset=["price_num", "rating_num"])

In [11]:
df.select("Name", "Category", "price_num", "rating_num", "rating_count_num").show(10)

+--------------------+-------------------+---------+----------+----------------+
|                Name|           Category|price_num|rating_num|rating_count_num|
+--------------------+-------------------+---------+----------+----------------+
|Redmi 10 Power (P...|tv, audio & cameras|  10999.0|       4.0|             965|
|OnePlus Nord CE 2...|tv, audio & cameras|  18999.0|       4.3|          113956|
|OnePlus Bullets Z...|tv, audio & cameras|   1999.0|       4.2|           90304|
|Samsung Galaxy M3...|tv, audio & cameras|  15999.0|       4.1|           24863|
|OnePlus Nord CE 2...|tv, audio & cameras|  18999.0|       4.3|          113956|
|Redmi 10 Power (S...|tv, audio & cameras|  10999.0|       4.0|             625|
|boAt Airdopes 141...|tv, audio & cameras|   1299.0|       3.9|          172347|
|Apple 20W USB-C P...|tv, audio & cameras|   1589.0|       4.6|           61748|
|Samsung Galaxy M3...|tv, audio & cameras|  15999.0|       4.1|           24863|
|realme narzo 50A ...|tv, au

In [12]:
#Analysis

In [13]:
#Top expensive products

In [14]:
df.orderBy("price_num", ascending=False).show(10)

+--------------------+-------------------+---------------+------+------------+---------+---------+----------+----------------+
|                Name|           Category|   Sub_Category|Rating|Rating_Count|    Price|price_num|rating_num|rating_count_num|
+--------------------+-------------------+---------------+------+------------+---------+---------+----------+----------------+
|Samsung Galaxy S2...|tv, audio & cameras|All Electronics|   4.4|          91|₹1,34,999| 134999.0|       4.4|              91|
|Apple iPhone 14 P...|tv, audio & cameras|All Electronics|   4.6|         306|₹1,32,999| 132999.0|       4.6|             306|
|Apple iPhone 14 P...|tv, audio & cameras|All Electronics|   4.5|         294|₹1,27,999| 127999.0|       4.5|             294|
|Apple iPhone 14 P...|tv, audio & cameras|All Electronics|   4.5|         294|₹1,27,999| 127999.0|       4.5|             294|
|Samsung Galaxy S2...|tv, audio & cameras|All Electronics|   4.4|          91|₹1,24,999| 124999.0|       4.4|  

In [15]:
#Top rated products

In [16]:
df.orderBy("rating_num", ascending=False).show(10)

+--------------------+-------------------+---------------+------+------------+------+---------+----------+----------------+
|                Name|           Category|   Sub_Category|Rating|Rating_Count| Price|price_num|rating_num|rating_count_num|
+--------------------+-------------------+---------------+------+------------+------+---------+----------+----------------+
|ZEBRONICS Zeb- Ha...|tv, audio & cameras|All Electronics|   5.0|           1|  ₹299|    299.0|       5.0|               1|
|Onsitego 1 Year E...|tv, audio & cameras|All Electronics|   5.0|           1|   ₹95|     95.0|       5.0|               1|
|ZEBRONICS Zeb- Ha...|tv, audio & cameras|All Electronics|   5.0|           1|  ₹299|    299.0|       5.0|               1|
|ZEBRONICS Zeb- Ha...|tv, audio & cameras|All Electronics|   5.0|           1|  ₹299|    299.0|       5.0|               1|
|Portronics Clean ...|tv, audio & cameras|All Electronics|   5.0|           7|  ₹499|    499.0|       5.0|               7|
|SD CASS

In [17]:
#Category-wise average price

In [18]:
df.groupBy("Category").avg("price_num").show()

+-------------------+------------------+
|           Category|    avg(price_num)|
+-------------------+------------------+
|tv, audio & cameras|2940.5372348962537|
+-------------------+------------------+



In [19]:
#Kafka

In [20]:
!wget https://archive.apache.org/dist/kafka/3.5.1/kafka_2.13-3.5.1.tgz
!tar -xzf kafka_2.13-3.5.1.tgz

--2026-04-14 14:52:32--  https://archive.apache.org/dist/kafka/3.5.1/kafka_2.13-3.5.1.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 106748875 (102M) [application/x-gzip]
Saving to: ‘kafka_2.13-3.5.1.tgz.1’

kafka_2.13-3.5.1.tg 100%[===================>] 101.80M  80.7KB/s    in 5m 49s  

2026-04-14 14:58:22 (299 KB/s) - ‘kafka_2.13-3.5.1.tgz.1’ saved [106748875/106748875]



In [21]:
!kafka_2.13-3.5.1/bin/zookeeper-server-start.sh -daemon kafka_2.13-3.5.1/config/zookeeper.properties

In [22]:
!kafka_2.13-3.5.1/bin/kafka-server-start.sh -daemon kafka_2.13-3.5.1/config/server.properties

In [23]:
!apt-get install openjdk-11-jdk -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-11-jdk is already the newest version (11.0.30+7-1ubuntu1~22.04).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [24]:
!kafka_2.13-3.5.1/bin/kafka-topics.sh --create \
--topic ecommerce-topic \
--bootstrap-server localhost:9092 \
--partitions 1 \
--replication-factor 1

Error while executing topic command : Topic 'ecommerce-topic' already exists.
[2026-04-14 14:58:30,845] ERROR org.apache.kafka.common.errors.TopicExistsException: Topic 'ecommerce-topic' already exists.
 (kafka.admin.TopicCommand$)


In [25]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 pyspark-shell'

In [26]:
#Sending Data to Kafka

In [27]:
!pip install kafka-python

In [29]:
df_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "ecommerce-topic") \
    .option("startingOffsets", "latest") \
    .load()

Py4JJavaError: An error occurred while calling o122.load.
: java.lang.NoClassDefFoundError: Could not initialize class org.apache.spark.sql.kafka010.KafkaSourceProvider$
	at org.apache.spark.sql.kafka010.KafkaSourceProvider.org$apache$spark$sql$kafka010$KafkaSourceProvider$$validateStreamOptions(KafkaSourceProvider.scala:338)
	at org.apache.spark.sql.kafka010.KafkaSourceProvider.sourceSchema(KafkaSourceProvider.scala:71)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceSchema(DataSource.scala:244)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceInfo$lzycompute(DataSource.scala:129)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceInfo(DataSource.scala:129)
	at org.apache.spark.sql.execution.streaming.StreamingRelation$.apply(StreamingRelation.scala:37)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$apply$1.applyOrElse(ResolveDataSource.scala:86)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$app

In [ ]:
from pyspark.sql.functions import col

df_stream = df_stream.selectExpr("CAST(value AS STRING)")

In [ ]:
query = df_stream.writeStream \
    .format("console") \
    .outputMode("append") \
    .start()

query.awaitTermination()

In [ ]:
from kafka import KafkaProducer
import json
import pandas as pd

# Load dataset
pdf = df.toPandas()

# Kafka producer
producer = KafkaProducer(
    bootstrap_servers='localhost:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

# Send limited fast data
for index, row in pdf.head(200).iterrows():
    data = row.to_dict()
    producer.send('ecommerce-topic', value=data)

producer.flush()
print("✅ Data Sent Successfully")